In [1]:
"""
Research assistant agents
"""

import dataclasses
import os
import typing
import xml.etree.ElementTree as ET
import requests

from dotenv import load_dotenv

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console

# load env
load_dotenv()

# Libraries

True

In [2]:
assert (api_key := os.environ.get("OPENAI_API_KEY")), (
    "Please ensure that you have set " "'OPENAI_API_KEY' in the environment."
)

model_client = OpenAIChatCompletionClient(
    model="gpt-6-luna",
    api_key=api_key,
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "gpt",
        "structured_output": True,
    },
)

model_client_for_tc = OpenAIChatCompletionClient(
    model="gpt-6-luna",
    api_key=api_key,
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "gpt",
        "structured_output": True,
    },
    reasoning_effort="none",
)

In [3]:
@dataclasses.dataclass
class EntryResult:
    id: int
    title: str
    summary: str
    link: str

    @classmethod
    def from_entry(cls, entry: ET.Element, entry_id: int) -> typing.Self:
        """Create an EntryResult instance from an arXiv XML entry."""
        return cls(id=entry_id, **EntryResult.extract_attrs_from(entry))

    @staticmethod
    def extract_attrs_from(entry: ET.Element) -> dict[str, str]:
        """Extracts the attributes for this class from an arXiv XML entry"""

        namespaces = {"atom": "http://www.w3.org/2005/Atom"}

        assert (
            title := typing.cast(ET.Element, entry.find("atom:title", namespaces)).text
        ), f"expected title, instead got: '{title}'"

        assert (
            summary := typing.cast(
                ET.Element, entry.find("atom:summary", namespaces)
            ).text
        ), f"expected summary, instead got: '{summary}'"

        assert (
            link := typing.cast(
                ET.Element,
                entry.find("atom:id", namespaces),
            ).text
        ), f"expected links, instead got: '{link}'"

        return {
            "title": title.strip(),
            "summary": summary.strip(),
            "link": link.strip(),
        }

    def to_json(self) -> dict[str, str | int]:
        """Return the entry data as a JSON-serializable dictionary."""
        return {
            "result": self.id,
            "title": self.title,
            "summary": self.summary,
            "link": self.link,
        }

    def __str__(self) -> str:
        return (
            f"{self.id}. "
            f"{self.title}\n   "
            f"Link: {self.link}\n   "
            f"Abstract: {self.summary[:300]}..."
        )


def arxiv_search_tool(query: str, max_results: int = 3) -> str:
    """Search arXiv for a query and return a formatted summary of results.

    Args:
        query: The topic to search for on arXiv.
        max_results: Maximum number of paper entries to include in the results.

    Returns:
        A newline-delimited summary of matching papers.
    """
    base_url = "http://export.arxiv.org/api/query"
    params = {"search_query": f"all:{query}", "start": 0, "max_results": max_results}

    # results = requests.get(base_url, params=params, timeout=10)
    results = requests.get(base_url, params=params)

    if results.status_code != 200:
        return f"Error: Unable to fetch data from arXiv (status {results.status_code})"

    root = ET.fromstring(results.text)
    entries = root.findall("atom:entry", {"atom": "http://www.w3.org/2005/Atom"})

    if not entries:
        return f"No papers found for topic '{query}'."

    return "\n".join(
        [f"Papers for '{query}':"]
        + [
            str(EntryResult.from_entry(entry, i))
            for i, entry in enumerate(entries, start=1)
        ]
    )

In [4]:
topic_agent = AssistantAgent(
    name="TopicAgent",
    description="Refines a broad research idea into a specific topic.",
    model_client=model_client,
    system_message="""
    ROLE: You are a Research Topic Refiner.
    TASK: Given a general idea, output one refined research topic.

    OUTPUT FORMAT:
    Refined Research Topic: <Topic>
    End with: TASK_COMPLETED_TOPIC_AGENT
    """,
)

In [5]:
topic_result = await topic_agent.run(task="AI in healthcare")
print(topic_result.messages[-1].content)

Refined Research Topic: Evaluating the accuracy and equity of AI-assisted diagnostic systems for detecting diabetic retinopathy across diverse patient populations.
TASK_COMPLETED_TOPIC_AGENT


In [10]:
arxiv_search_tool(topic_result.messages[-1].content, 1)

"Papers for 'Refined Research Topic: Evaluating the accuracy and equity of AI-assisted diagnostic systems for detecting diabetic retinopathy across diverse patient populations.\nTASK_COMPLETED_TOPIC_AGENT':\n1. Mini Lesions Detection on Diabetic Retinopathy Images via Large Scale CNN Features\n   Link: http://arxiv.org/abs/1911.08588v1\n   Abstract: Diabetic retinopathy (DR) is a diabetes complication that affects eyes. DR is a primary cause of blindness in working-age people and it is estimated that 3 to 4 million people with diabetes are blinded by DR every year worldwide. Early diagnosis have been considered an effective way to mitigate such..."

In [6]:
paper_agent = AssistantAgent(
    name="PaperAgent",
    model_client=model_client_for_tc,
    description="Finds research papers for the given topic.",
    tools=[arxiv_search_tool],
    system_message="""
    ROLE: You are a Paper Discovery Agent.
    TASK: Find 2–3 recent research papers related to the topic.

    OUTPUT FORMAT:
    ### Topic: <Topic>
    - Paper 1: <Title> — <Summary> - <URL>
    - Paper 2: <Title> — <Summary> - <URL>
    """,
)

In [13]:
paper_result = await paper_agent.run(task=topic_result.messages[-1].content)
print(paper_result.messages[-1].content)

Papers for 'diabetic retinopathy artificial intelligence screening generalizability race ethnicity fairness validation':
1. The Artificial Scientist: Logicist, Emergentist, and Universalist Approaches to Artificial General Intelligence
   Link: http://arxiv.org/abs/2110.01831v1
   Abstract: We attempt to define what is necessary to construct an Artificial Scientist, explore and evaluate several approaches to artificial general intelligence (AGI) which may facilitate this, conclude that a unified or hybrid approach is necessary and explore two theories that satisfy this requirement to s...
2. Mobile-Ready Automated Triage of Diabetic Retinopathy Using Digital Fundus Images
   Link: http://arxiv.org/abs/2602.21943v1
   Abstract: Diabetic Retinopathy (DR) is a major cause of vision impairment worldwide. However, manual diagnosis is often time-consuming and prone to errors, leading to delays in screening. This paper presents a lightweight automated deep learning framework for efficient ass

In [ ]:
insight_agent = AssistantAgent(
    name="InsightAgent",
    model_client=model_client,
    description="Summarizes key insights from discovered papers.",
    system_message="""
    ROLE: You are an Insight Synthesizer.
    TASK: Provide 3–5 bullet points summarizing common trends or findings.

    OUTPUT FORMAT:
    Key Insights:
    - <Insight 1>
    - <Insight 2>
    """,
)

In [ ]:
insight_result = await insight_agent.run(task=paper_result.messages[-1].content)
print(insight_result.messages[-1].content)

Key Insights:
- The results include two papers directly related to diabetic retinopathy: one on mobile-ready automated triage and one on detecting small lesions in fundus images.
- The mobile-ready paper focuses on efficient severity assessment; the lesion-detection paper emphasizes early detection. The provided abstracts do not establish how well either approach generalizes across settings.
- None of the supplied abstracts explicitly reports validation by race or ethnicity, subgroup fairness analyses, or performance disparities.
- Several results are unrelated to diabetic retinopathy or AI screening, so this search has limited relevance to the fairness and generalizability question.


In [16]:
report_agent = AssistantAgent(
    name="ReportAgent",
    description="Compiles a concise research report.",
    model_client=model_client,
    system_message="""
    ROLE: You are a Report Compiler.
    TASK: Create a mini research report.

    OUTPUT FORMAT:
    ## Research Report
    ### Introduction
    ...
    ### Key Findings
    ...
    ### Discussion
    ...
    ### Conclusion
    ...
    """,
)

In [ ]:
report_result = await report_agent.run(task=insight_result.messages[-1].content)
print(report_result.messages[-1].content)

## Research Report

### Introduction
The supplied results include two papers directly related to diabetic retinopathy: one on mobile-ready automated triage and another on detecting small lesions in fundus images. This report summarizes their relevance to screening, fairness, and generalizability based on the provided information.

### Key Findings
- The mobile-ready paper focuses on efficient assessment of disease severity.
- The lesion-detection paper emphasizes early detection of small lesions in fundus images.
- The provided abstracts do not establish how well either approach generalizes across different settings.
- None of the supplied abstracts explicitly reports validation by race or ethnicity, subgroup fairness analyses, or performance disparities.
- Several other results are unrelated to diabetic retinopathy or AI screening, limiting the search’s relevance to questions of fairness and generalizability.

### Discussion
The two relevant papers address complementary screening task

In [18]:
gap_agent = AssistantAgent(
    name="GapAgent",
    model_client=model_client,
    description="Performs research gap analysis and suggests future directions.",
    system_message="""
    ROLE: You are a Research Gap Analyst.
    TASK: Identify 2–3 research gaps and 2–3 future directions.

    OUTPUT FORMAT:
    Research Gaps:
    - <Gap 1>
    - <Gap 2>

    Future Directions:
    - <Direction 1>
    - <Direction 2>

    FINAL REPORT
    """,
)

In [ ]:
gap_result = await gap_agent.run(task=report_result.messages[-1].content)
print(gap_result.messages[-1].content)

Research Gaps:
- The supplied summaries do not establish whether either approach generalizes across populations, clinical settings, or imaging devices.
- They report no race- or ethnicity-stratified performance or other subgroup fairness analyses, leaving potential disparities unknown.
- The limited set of relevant studies addresses different tasks—severity triage and small-lesion detection—so their comparative clinical utility is unclear.

Future Directions:
- Conduct external and prospective validation across diverse populations, care settings, and fundus-camera types, reporting performance by relevant subgroups.
- Evaluate both approaches in screening workflows, including their effects on referral decisions, missed disease, and access to follow-up care.

FINAL REPORT: The supplied evidence identifies relevant AI approaches but is insufficient to assess their generalizability, fairness, or comparative value in clinical screening.


In [21]:
text_mention_termination = TextMentionTermination("FINAL REPORT")
max_messages_termination = MaxMessageTermination(max_messages=10)
termination = max_messages_termination | text_mention_termination

In [22]:
team = RoundRobinGroupChat(
    name="AgentGroup",
    participants=[topic_agent, paper_agent, insight_agent, report_agent, gap_agent],
    termination_condition=termination,
)

In [ ]:
# await Console(team.run_stream(task="AI for healthcare"))

---------- TextMessage (user) ----------
AI for healthcare
---------- TextMessage (TopicAgent) ----------
Refined Research Topic: Assessing the effectiveness and equity of AI-based clinical decision support in detecting sepsis early in adult hospital patients.  
TASK_COMPLETED_TOPIC_AGENT
---------- ToolCallRequestEvent (PaperAgent) ----------
[FunctionCall(id='call_j6tp17JVkUltseiVmP7ot8RN', arguments='{"query":"AI machine learning early sepsis detection clinical decision support hospital patients fairness equity prospective evaluation","max_results":5}', name='arxiv_search_tool')]
---------- ToolCallExecutionEvent (PaperAgent) ----------
[FunctionExecutionResult(content="Papers for 'AI machine learning early sepsis detection clinical decision support hospital patients fairness equity prospective evaluation':\n1. Identification of Pediatric Sepsis Subphenotypes for Enhanced Machine Learning Predictive Performance: A Latent Profile Analysis\n   Link: http://arxiv.org/abs/1908.09038v1\n

TaskResult(messages=[TextMessage(id='c26debc2-8017-4a00-b6f7-a90a7b7f9da5', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 10, 11, 16, 50, 51, 919583, tzinfo=datetime.timezone.utc), content='AI for healthcare', type='TextMessage'), TextMessage(id='bec0eeb4-c836-42db-a0e0-b1399e909454', source='TopicAgent', models_usage=RequestUsage(prompt_tokens=108, completion_tokens=77), metadata={}, created_at=datetime.datetime(2026, 10, 11, 16, 50, 54, 128517, tzinfo=datetime.timezone.utc), content='Refined Research Topic: Assessing the effectiveness and equity of AI-based clinical decision support in detecting sepsis early in adult hospital patients.  \nTASK_COMPLETED_TOPIC_AGENT', type='TextMessage'), ToolCallRequestEvent(id='fcff15db-ff7d-4978-8a6f-84ea8ffa6279', source='PaperAgent', models_usage=RequestUsage(prompt_tokens=1008, completion_tokens=39), metadata={}, created_at=datetime.datetime(2026, 10, 11, 16, 50, 55, 461004, tzinfo=datetime.timezone.utc), cont

In [26]:
user_proxy_agent = UserProxyAgent(
    name="UserProxyAgent",
    description="A proxy for the user to approve or disapprove tasks.",
)

In [27]:
user_approved = False


def selector_func_with_user_proxy(
    messages: typing.Sequence[BaseAgentEvent | BaseChatMessage],
) -> str | None:
    global user_approved

    # if first message, start with topic agent
    if len(messages) == 0:
        return topic_agent.name

    last_source = messages[-1].source

    # if it does not require user's input, let the agent talk automatically
    if last_source != user_proxy_agent.name:
        return None  # means use default sequencing in SelectorGroupChat

    # if user just responded (approved or reject)
    content = messages[-1].content.upper()
    if "APPROVED" in content:
        user_approved = True
        # proceed to the next logical agent (e.g., paper_agent)
        return paper_agent.name

    # if disapproved, return to topic_agent for revision
    return topic_agent.name

In [ ]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the topic agent has assigned tasks before other agents start working.
Only select one agent.
"""

task = "AI for healthcare"

# Run the chat again with the user proxy agent and selector function.
team = SelectorGroupChat(
    [
        topic_agent,
        paper_agent,
        insight_agent,
        report_agent,
        gap_agent,
        user_proxy_agent,
    ],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    selector_func=selector_func_with_user_proxy,
    allow_repeated_speaker=False,
)

In [29]:
await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
AI for healthcare
---------- TextMessage (TopicAgent) ----------
Refined Research Topic: Evaluating whether AI-based tools can improve early detection of sepsis in emergency departments without worsening diagnostic disparities across patient groups.  
TASK_COMPLETED_TOPIC_AGENT
---------- ToolCallRequestEvent (PaperAgent) ----------
[FunctionCall(id='call_FOYsSkp47hqZwfcD785ggxl5', arguments='{"query":"emergency department sepsis early detection artificial intelligence machine learning diagnostic disparities racial equity","max_results":5}', name='arxiv_search_tool')]
---------- ToolCallExecutionEvent (PaperAgent) ----------
[FunctionExecutionResult(content='Papers for \'emergency department sepsis early detection artificial intelligence machine learning diagnostic disparities racial equity\':\n1. Identification of Pediatric Sepsis Subphenotypes for Enhanced Machine Learning Predictive Performance: A Latent Profile Analysis\n   Link: http://arxi

TaskResult(messages=[TextMessage(id='e990d447-b036-4c75-952b-4fd96d759da4', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 10, 11, 17, 35, 54, 267145, tzinfo=datetime.timezone.utc), content='AI for healthcare', type='TextMessage'), TextMessage(id='e1dcf944-e7f7-4323-8b07-929acfb4489d', source='TopicAgent', models_usage=RequestUsage(prompt_tokens=155, completion_tokens=59), metadata={}, created_at=datetime.datetime(2026, 10, 11, 17, 35, 57, 907593, tzinfo=datetime.timezone.utc), content='Refined Research Topic: Evaluating whether AI-based tools can improve early detection of sepsis in emergency departments without worsening diagnostic disparities across patient groups.  \nTASK_COMPLETED_TOPIC_AGENT', type='TextMessage'), ToolCallRequestEvent(id='65c7049f-0d25-4842-a221-8168cef34fce', source='PaperAgent', models_usage=RequestUsage(prompt_tokens=1584, completion_tokens=38), metadata={}, created_at=datetime.datetime(2026, 10, 11, 17, 36, 1, 226395, tzinfo